In [ ]:
import sys
import shutil
import importlib
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from keras.layers import TFSMLayer
from google.colab import files
import os

In [ ]:
import sys
sys.path.append('/content')

import myNN
import importlib; importlib.reload(myNN)

print(dir(myNN))  # checking stuff


['ZScoreLayer', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__warningregistry__', 'h5py', 'intList', 'layerNum', 'loadWeights', 'load_model', 'model', 'os', 'tf']


In [ ]:
# INSPECTING MODEL EXPORTED FROM MATLAB network to tensorflow function

# Access the model object
import myNN.model as model

net = model.create_model()
net.summary()

print("Input shape:", net.input_shape)  # should be (None, 1024)
print("Output shape:", net.output_shape)  # should be (None, 3)


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_unnormalized (InputLayer) │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ input_ (ZScoreLayer)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc_1_ (Dense)                   │ (None, 189)            │       193,725 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 189)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc_2_ (Dense)                   │ (None, 189)            │        35,910 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 189)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc_3_ (Dense)                   │ (None, 3)              │           570 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax_2 (Softmax)             │ (None, 3)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 230,205 (899.24 KB)

 Trainable params: 230,205 (899.24 KB)

 Non-trainable params: 0 (0.00 B)

Input shape: (None, 1024)
Output shape: (None, 3)


In [ ]:
import h5py

# Open the h5 file to inspect its structure
with h5py.File('/content/myNN/weights.h5', 'r') as f:
    # List all groups in the file
    print("Keys in weights.h5:", list(f.keys()))

    # Optionally, explore the first layer or a specific layer
    layer_name = list(f.keys())[0]  # Adjust if needed
    print(f"Structure of layer {layer_name}:")
    print(f[layer_name].attrs)


Keys in weights.h5: ['fc_1_', 'fc_2_', 'fc_3_', 'input_']
Structure of layer fc_1_:
<Attributes of HDF5 object at 137863453157040>


In [ ]:
with h5py.File('/content/myNN/weights.h5', 'r') as f:
    # List all groups (layers) in the file
    for group_name in f:
        group = f[group_name]
        print(f"Layer: {group_name}")
        print("    Number of variables:", group.attrs.get('NumVars'))
        print("    Weights:", list(group.keys()))


Layer: fc_1_
    Number of variables: [2.]
    Weights: ['bias', 'kernel']
Layer: fc_2_
    Number of variables: [2.]
    Weights: ['bias', 'kernel']
Layer: fc_3_
    Number of variables: [2.]
    Weights: ['bias', 'kernel']
Layer: input_
    Number of variables: [2.]
    Weights: ['mean', 'stdev']


In [ ]:
model = myNN.load_model()

print("Mean shape:", model.get_layer("input_").mean.shape)
print("Mean (first 5):", model.get_layer("input_").mean.numpy()[:5])

model.export('/content/my_saved_NN')

Mean shape: (1024,)
Mean (first 5): [ 71.05658  75.53707  79.59717  87.02772 107.62675]


/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_unnormalized']
Received: inputs=Tensor(shape=(None, 1024))
  warnings.warn(msg)


Saved artifact at '/content/my_saved_NN'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1024), dtype=tf.float32, name='input_unnormalized')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  137862535338448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137862535338064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137862535339792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137862535340368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137862535342288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137862535340560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137862535339984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137862535342864: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [ ]:
# Print summary
model.summary()

# input info
print("Input name:", model.inputs[0].name)
print("Input shape:", model.inputs[0].shape)

# output info
print("Output name:", model.outputs[0].name)
print("Output shape:", model.outputs[0].shape)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_unnormalized (InputLayer) │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ input_ (ZScoreLayer)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc_1_ (Dense)                   │ (None, 189)            │       193,725 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 189)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc_2_ (Dense)                   │ (None, 189)            │        35,910 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 189)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc_3_ (Dense)                   │ (None, 3)              │           570 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (None, 3)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 230,205 (899.24 KB)

 Trainable params: 230,205 (899.24 KB)

 Non-trainable params: 0 (0.00 B)

Input name: input_unnormalized
Input shape: (None, 1024)
Output name: keras_tensor_6
Output shape: (None, 3)


In [ ]:
# NOTE: This part of the code is to create a .raw file from the test/train data to be used
# as calibration data for quantization in Qualcomm's Neural Processing SDK.

# SKIP THIS STEP IF you don't need this.

# Load the CSV
df = pd.read_csv("test_7_3label_incorrect.csv")

# Drop the label column
input_data = df.iloc[:, :-1].astype(np.float32).values

# Create output folder
os.makedirs("calib_raw_inputs_incorrect3", exist_ok=True)

# Write each input as .raw and list in input_list.txt
with open("input_list.txt", "w") as f:
    for i, row in enumerate(input_data):
        filepath = f"calib_raw_inputs_incorrect3/input_{i}.raw"
        row.astype(np.float32).reshape(1, -1).tofile(filepath)
        f.write(os.path.abspath(filepath) + "\n")

# To create the input_list.txt , run this locally:
# find "$(pwd)/calib_raw_inputs" -name "*.raw" | sort | awk '{ print }' > input_list.txt
# change the dir name accordingly obv
# do NOT use the input list file generated here

# Archive and download
shutil.make_archive("calib_inputs_incorrect3", 'zip', "calib_raw_inputs_incorrect3")
files.download("calib_inputs_incorrect3.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# test
test_raw = np.fromfile("calib_raw_inputs_incorrect3/input_0.raw", dtype=np.float32).reshape(1, 1024)
print(test_raw.shape)  # should print (1, 1024)

(1, 1024)


In [ ]:
# Investigating the SNPE generated raw files, i.e., output of snpe-net-run

# Directory containing raw output files from SNPE
output_dir = "snpe_output_quantized"  # Change this to your actual path

# List all .raw files
raw_files = sorted([f for f in os.listdir(output_dir) if f.endswith(".raw")])

# Loop through and read each file
for filename in raw_files:
    file_path = os.path.join(output_dir, filename)
    output = np.fromfile(file_path, dtype=np.float32)
    print(f"{filename}: {output}")

StatefulPartitionedCall_1:0.raw: [1.0949009e-05 7.4948553e-06 9.9998152e-01]


In [ ]:
!zip -r my_saved_NN.zip my_saved_NN

files.download("my_saved_NN.zip")

  adding: my_saved_NN/ (stored 0%)
  adding: my_saved_NN/fingerprint.pb (stored 0%)
  adding: my_saved_NN/saved_model.pb (deflated 84%)
  adding: my_saved_NN/variables/ (stored 0%)
  adding: my_saved_NN/variables/variables.index (deflated 59%)
  adding: my_saved_NN/variables/variables.data-00000-of-00001 (deflated 8%)
  adding: my_saved_NN/assets/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Print mean and std from ZScoreLayer
zlayer = None
for layer in model.layers:
    if isinstance(layer, myNN.ZScoreLayer):
        zlayer = layer
        break

if zlayer:
    print("Found ZScoreLayer")
    print("Mean:", zlayer.mean.numpy())
    print("Std:", zlayer.std.numpy())
else:
    print("ZScoreLayer not found in model.")

Found ZScoreLayer
Mean: [ 71.05658   75.53707   79.59717  ...  95.35049  105.896065  84.649895]
Std: [24.894312 27.084814 26.780912 ... 35.477444 41.457195 31.845758]


In [ ]:
!saved_model_cli show --dir my_saved_NN --all

2025-06-05 18:23:23.747206: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749147803.769764    1341 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749147803.776357    1341 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-05 18:23:29.500638: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)

MetaGraphDef with tag-set: 'serve' contains the following SignatureDefs:

signature_def['__saved_model_init_op']:
  The given SavedModel SignatureDef contains the following input(s):
  The given SavedModel SignatureDef contains the following output(s):
 

In [ ]:
# === Load and Preprocess Test Data ===
test_df = pd.read_csv('/content/test_7_3label.csv')  # ← Update path if needed

# Separate features and labels
X_test = test_df[[f'var{i}' for i in range(1, 1025)]].values
y_test = test_df['three_label'].values

# Encode labels to integers (e.g., 'correct' → 1, 'incorrect' → 0, 'not_sitting' → 2)
label_map = {'correct': 0, 'incorrect': 1, 'not_sitting': 2}
y_test_encoded = np.array([label_map[label] for label in y_test])

# === Run Inference ===
y_pred_probs = model.predict({'input_unnormalized': X_test})
y_pred_classes = np.argmax(y_pred_probs, axis=1)  # Assumes model output is softmax

# === Evaluate ===
print("Accuracy:", accuracy_score(y_test_encoded, y_pred_classes))
print("Confusion Matrix:\n", confusion_matrix(y_test_encoded, y_pred_classes))
print("Classification Report:\n", classification_report(
    y_test_encoded,
    y_pred_classes,
    target_names=['incorrect', 'correct', 'not_sitting']
))


258/258 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Accuracy: 0.8959825221507465
Confusion Matrix:
 [[1625  299    0]
 [ 558 5338    0]
 [   0    0  419]]
Classification Report:
               precision    recall  f1-score   support

   incorrect       0.74      0.84      0.79      1924
     correct       0.95      0.91      0.93      5896
 not_sitting       1.00      1.00      1.00       419

    accuracy                           0.90      8239
   macro avg       0.90      0.92      0.91      8239
weighted avg       0.90      0.90      0.90      8239



In [ ]:
# QUANTIZING TFLITE
import tensorflow as tf

# Path to your SavedModel directory
saved_model_dir = "/content/my_saved_NN"

# Converter
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)

# (Optional) Enable optimizations like quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Convert the model
tflite_model = converter.convert()

# Save it to a file
with open("model.tflite", "wb") as f:
    f.write(tflite_model)


In [ ]:
# NOT QUANTIZING TFLITE

# USE THIS if wanting to use snpe-tflite-to-dc
import tensorflow as tf

saved_model_dir = "/content/my_saved_NN"

converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)

# Set fixed shape: batch size 1, 1024 features
converter.experimental_new_converter = True
converter._experimental_lower_tensor_list_ops = False
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
converter.optimizations = []

# This is the fix: set shape for the input explicitly
def representative_dataset_gen():
    for _ in range(100):
        yield [tf.random.uniform([1, 1024], dtype=tf.float32)]

converter.representative_dataset = representative_dataset_gen

# Force input shape
converter._experimental_input_shapes = {'input_unnormalized': [1, 1024]}

tflite_model = converter.convert()

with open("model_fixed_dims.tflite", "wb") as f:
    f.write(tflite_model)

interpreter = tf.lite.Interpreter(model_path="model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input dtype:", input_details[0]['dtype'])
print("Output dtype:", output_details[0]['dtype'])


Input dtype: <class 'numpy.float32'>
Output dtype: <class 'numpy.float32'>


In [ ]:
# ================== TESTING TFLITE MODEL ======================

# Load TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path="model_fixed_dims.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# NOTE: The names below will likely be wrong.
# Use Netron to get the correct names.

print("Input shape:", input_details[0]['shape'])
print("Input name:", input_details[0]['name'])

print("Output shape:", output_details[0]['shape'])
print("Output name:", output_details[0]['name'])

Input shape: [   1 1024]
Input name: serving_default_input_unnormalized:0
Output shape: [1 3]
Output name: StatefulPartitionedCall_1:0


In [ ]:
correct = 0
total = len(X_test)

for i in range(total):
    input_data = X_test[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    pred = np.argmax(output)

    if pred == y_test_encoded[i]:
        correct += 1

print("TFLite accuracy:", correct / total)


TFLite accuracy: 0.8952542784318486
